# Methodology Investigation: Iterative Refinement of the Plus/Minus Metric

## Background

In this project, team draft efficiency is measured using a **plus/minus** metric — the difference between a player's actual draft pick number and their hindsight rank based on NHL performance.

Hindsight ranks are calculated per draft year using a Performance Score formula:

`Performance Score = Games Played + (Points × 2.29)`

Players are ranked within each draft year by their Performance Score, and each team's picks are evaluated against that hindsight order. The sum of a team's plus/minus scores across all picks and all drafts produces their final draft efficiency rating.

## The Problem: A Two-Pass Solution

After applying the Performance Score formula and building hindsight rankings, we identified two structural flaws in the plus/minus calculation that were systematically distorting team scores. Both problems stem from the same root cause — the hindsight ranking assigns every non-contributor the same rank, creating phantom space in the draft order that inflates plus/minus scores for late round picks.

We address this with a two-pass system:

**Pass 1 — Remove the noise:** Players who never established a meaningful NHL career are excluded from contributing positive plus/minus to their team's score. Teams are still penalized for burning early picks on busts, but late round non-contributors are neutralized.

**Pass 2 — Compress the scale:** The plus/minus calculation is rebuilt using only meaningful contributors, eliminating the phantom space entirely and naturally bounding the scale.

## Revealing the Problem

The plus/minus metric is built on hindsight rankings. For each draft year, every player is ranked by their Performance Score from best to worst. A player's plus/minus is then calculated as:

`plus/minus = actual pick number − hindsight rank`

This works well for players who had real NHL careers. But it breaks down for players who never played a game — or barely played at all. All players with a Performance Score of 0 share the same hindsight rank: the first position after the last player who actually played in the NHL. In a draft where 95 players made the NHL, every zero-score player is assigned hindsight rank 96.

This creates a mechanical flaw. Any player drafted after position 95 who never played a game receives a positive plus/minus by default — not because they were a good pick, but simply because their pick number exceeds their shared hindsight rank.

The cells below demonstrate this concretely.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('../data/outputs/draft_scores_pre_fix.csv')

# Show zero performance score players sorted by plus/minus
zeros = (df[df['performance_score'] == 0]
         [['player_name', 'drafted_by', 'draft_year', 'round', 'pick_number', 'performance_score', 'plus_minus']]
         .sort_values('plus_minus', ascending=False)
         .head(20))

print(zeros.to_string())

              player_name    drafted_by  draft_year  round  pick_number  performance_score  plus_minus
393        Justin Krueger      Carolina        2006      7          213                0.0         134
392           Logan Pyett       Detroit        2006      7          212                0.0         133
389          Per Johnsson       Calgary        2006      7          209                0.0         130
388         Kyell Henegan    New Jersey        2006      7          208                0.0         129
205       Philippe Paquet      Montreal        2005      7          229                0.0         128
387      Benjamin Breault       Buffalo        2006      7          207                0.0         128
386         Viktor Sjodin     Nashville        2006      7          206                0.0         127
203          Andrew Orpik       Buffalo        2005      7          227                0.0         126
385          Andrei Popov  Philadelphia        2006      7          205  

The players above never played a single NHL game — yet they show positive plus/minus values as high as 134. This is not because they were good picks. It is an artifact of how hindsight ranks are assigned.

The cutoff — the pick number at which this inflation begins — is determined entirely by how many players from each draft happened to make the NHL. This varies significantly year to year.

In [2]:
# NHL player count per draft year — this is the cutoff point
cutoff = (df[df['performance_score'] > 0]
          .groupby('draft_year')['hindsight_rank']
          .max()
          .reset_index()
          .rename(columns={'hindsight_rank': 'nhl_player_count'}))

print("NHL players per draft year (the cutoff):")
print(cutoff.to_string())

NHL players per draft year (the cutoff):
    draft_year  nhl_player_count
0         2005                95
1         2006                75
2         2007                91
3         2008                90
4         2009               101
5         2010                91
6         2011               116
7         2012                95
8         2013                95
9         2014                90
10        2015               100
11        2016               107
12        2017                90


The cutoff varies from 75 players in 2006 to 116 in 2011. Any player drafted after pick 75 in 2006, or after pick 116 in 2011, who never played a game receives a positive plus/minus by default. The cutoff has nothing to do with draft strategy — it is determined entirely by how many players happened to make the NHL that year.

This is the phantom space. In a 230-pick draft where only 95 players made the NHL, picks 96 through 230 create 135 slots of empty space. Late round picks who contributed nothing can leap over this entire graveyard of zeros and generate massive positive plus/minus scores for their teams.

But the problem does not stop at zero. Players who technically appeared in an NHL game but contributed almost nothing suffer from the same distortion — they rank just above the zeros, generating large positive plus/minus scores simply by being drafted late.

In [4]:
# Show low contributors with positive plus/minus
# These players played some NHL games but contributed almost nothing
low_contributors = df[
    (df['performance_score'] > 0) &
    (df['performance_score'] < 100) &
    (df['plus_minus'] > 0)
][['player_name', 'drafted_by', 'draft_year', 'round', 'games_played', 'points', 'performance_score', 'plus_minus']].sort_values('plus_minus', ascending=False).head(20)

print(low_contributors.to_string())

              player_name    drafted_by  draft_year  round  games_played  points  performance_score  plus_minus
204              Chad Rau       Toronto        2005      7           9.0     2.0              13.58         147
381          Arturs Kulda      Winnipeg        2006      7          15.0     2.0              19.58         141
188          Ryan Russell    NY Rangers        2005      7          41.0     2.0              45.58         140
390          Will O'Neill      Winnipeg        2006      7           1.0     0.0               1.00         135
2478      Ivan Chekhovich      San Jose        2017      7           4.0     1.0               6.29         131
364            Jan Mursak       Detroit        2006      6          46.0     4.0              55.16         128
1526          Viktor Loov       Toronto        2012      7           4.0     2.0               8.58         125
2463        Wyatt Kalynuk  Philadelphia        2017      7          26.0     9.0              46.61     

Players like these — who played under 100 NHL games over careers that spanned nearly a decade — are generating plus/minus scores of 80, 100, or more for their teams. They are indistinguishable from true busts in any meaningful hockey sense, yet the methodology treats them very differently from zero-score players.

The root cause is the same in both cases: phantom space created by the hindsight ranking system. Whether a player scored 0 or 100 on the Performance Score scale, if they were drafted late enough, they leap over a graveyard of non-contributors and generate unearned credit for their team.

This requires a unified fix.

## Pass 1 — Neutralizing Non-Contributors

The unified fix addresses both zero-score players and near-zero contributors with a single rule, grounded in a clear and defensible threshold:

**Any player with fewer than 200 games played who was drafted after the cutoff for their draft year contributes 0 plus/minus.**

200 games is roughly 2.5 full NHL seasons. Given that our dataset covers drafts from 2005 to 2017 — meaning even the most recent draftees have had nearly a decade to establish themselves — a player who has not reached 200 games has not made it in any meaningful sense.

The asymmetry is intentional. Teams that burned a high pick on a player who never stuck around should still be penalized — that is a real investment of draft capital that produced nothing. But a 6th round pick at position 180 who played 15 games should not be rewarding their team with 100 plus/minus points simply for being drafted late.

The fix preserves accountability where it matters and removes noise where it does not.

In [14]:
GP_THRESHOLD = 200

df_pass1 = df.copy()
df_pass1 = df_pass1.merge(cutoff, on='draft_year')

# Zero out plus/minus for non-contributors drafted after the cutoff
mask_after_cutoff = (
    (df_pass1['games_played'] < GP_THRESHOLD) &
    (df_pass1['pick_number'] > df_pass1['nhl_player_count'])
)
df_pass1.loc[mask_after_cutoff, 'plus_minus'] = 0

# Zero out positive plus/minus for non-contributors drafted before the cutoff
# They keep their negative (wasted draft capital) but cannot contribute positives
mask_positive_before_cutoff = (
    (df_pass1['games_played'] < GP_THRESHOLD) &
    (df_pass1['pick_number'] <= df_pass1['nhl_player_count']) &
    (df_pass1['plus_minus'] > 0)
)
df_pass1.loc[mask_positive_before_cutoff, 'plus_minus'] = 0

# Quantify the impact
total_zeroed = mask_after_cutoff.sum() + mask_positive_before_cutoff.sum()
points_eliminated = (
    df[mask_after_cutoff]['plus_minus'].sum() + 
    df[mask_positive_before_cutoff]['plus_minus'].sum()
)
total_positive = df[df['plus_minus'] > 0]['plus_minus'].sum()
pct_eliminated = round((points_eliminated / total_positive) * 100, 1)

print(f"GP threshold: {GP_THRESHOLD} games")
print(f"Players zeroed out after cutoff: {mask_after_cutoff.sum()}")
print(f"Non-contributors with positive plus/minus zeroed out: {mask_positive_before_cutoff.sum()}")
print(f"Total players zeroed out: {total_zeroed}")
print(f"Total artificial plus/minus eliminated: {points_eliminated}")
print(f"Total positive plus/minus before Pass 1: {total_positive}")
print(f"Percentage of total positive plus/minus eliminated: {pct_eliminated}%")

GP threshold: 200 games
Players zeroed out after cutoff: 1185
Non-contributors with positive plus/minus zeroed out: 64
Total players zeroed out: 1249
Total artificial plus/minus eliminated: 76360
Total positive plus/minus before Pass 1: 102777
Percentage of total positive plus/minus eliminated: 74.3%


73.2% of all positive plus/minus in the dataset was artificial inflation — not draft skill, not scouting acumen, just noise generated by the phantom space in the hindsight ranking system. Nearly three quarters of the positive scores that were driving team rankings had no legitimate basis.

This is not a minor calibration. The original metric was fundamentally broken. Pass 1 eliminates the noise. Pass 2 ensures it cannot return.

## Pass 2 — Compressing the Scale

In [15]:
# Show the asymmetry after Pass 1
# Highest plus/minus vs lowest plus/minus
top_plus = df_pass1.nlargest(15, 'plus_minus')[
    ['player_name', 'drafted_by', 'draft_year', 'round', 'pick_number', 'games_played', 'performance_score', 'plus_minus']
]

top_minus = df_pass1.nsmallest(15, 'plus_minus')[
    ['player_name', 'drafted_by', 'draft_year', 'round', 'pick_number', 'games_played', 'performance_score', 'plus_minus']
]

print("Top 15 highest plus/minus after Pass 1:")
print(top_plus.to_string())
print("\nTop 15 lowest plus/minus after Pass 1:")
print(top_minus.to_string())

Top 15 highest plus/minus after Pass 1:
           player_name   drafted_by  draft_year  round  pick_number  games_played  performance_score  plus_minus
206   Patric Hornqvist    Nashville        2005      7          230         901.0            2144.47         219
193     Anton Stralman      Toronto        2005      7          216         938.0            1608.97         201
1339      Ondrej Palat    Tampa Bay        2011      7          208         889.0            2077.51         194
1714  MacKenzie Weegar      Florida        2013      7          206         614.0            1239.17         178
391        Erik Condra       Ottawa        2006      7          211         372.0             598.71         177
576       Justin Braun     San Jose        2007      7          201         842.0            1297.71         176
1904        Jake Evans     Montreal        2014      7          207         400.0             738.92         172
177   Sergei Kostitsyn     Montreal        2005      7  

The asymmetry is immediately visible. The highest plus/minus scores — all 7th round picks — reach as high as +219, while the lowest scores bottom out around -95. The ceiling is more than double the floor.

This is structural, not coincidental. A player drafted 230th can mathematically generate a plus/minus as high as +229 if they rank first in hindsight. But a player drafted 1st overall can only generate a plus/minus as low as roughly -94, since the hindsight ranking floor is determined by how many players made the NHL that year — typically around 90-116.

The result is a metric that rewards late round luck far more than it penalizes early round failure. Patric Hornqvist, drafted 230th in 2005, nets Nashville +219. Mark McNeill, a wasted 18th overall pick in 2011, costs Chicago only -95. The scales are fundamentally unbalanced.

Pass 2 fixes this by rebuilding the plus/minus calculation from scratch using only meaningful contributors — eliminating the phantom space entirely and naturally bounding the scale on both sides.

### How Pass 2 Works

Pass 2 rebuilds the plus/minus calculation from scratch for meaningful contributors — players with 200 or more games played. Every other player already has their score determined by Pass 1.

The final plus/minus for every player in the dataset falls into exactly one of three categories:

**1. Zero** — players with under 200 games drafted after the cutoff

These were speculative late round picks that didn't pan out. They contribute nothing in either direction — no reward, no penalty.

**2. Original negative** — players with under 200 games drafted before the cutoff

These players burned real draft capital and delivered nothing meaningful. Their penalty is preserved from the original calculation. Teams are held accountable for early round busts.

**3. Compressed plus/minus** — players with 200+ games played

The re-ranking is built exclusively from this population. Their actual draft positions are re-numbered 1 through N in the order they were drafted, and they are re-ranked 1 through N by performance score. Plus/minus is then calculated within this compressed space.

### Why This Is Elegant

The compressed re-ranking solves the asymmetry problem without introducing any arbitrary external cap. The scale bounds itself naturally — in a draft where 95 players made the NHL as meaningful contributors, the maximum possible plus/minus in either direction is roughly ±94. The floor and ceiling emerge from the data itself, the same way the cutoff did in Pass 1.

This also changes what the metric is actually measuring. The original plus/minus asked: "how did this pick compare to every player drafted, including 130 players who never made it?" The compressed plus/minus asks a sharper, more honest question: "among the players from this draft who actually had NHL careers, how well did each team identify them?"

That is the question this project set out to answer. Pass 2 makes sure the metric is actually answering it.

In [16]:
# Pass 2 — Compressed re-ranking for meaningful contributors
CONTRIBUTOR_THRESHOLD = 200

# Work from the Pass 1 dataframe
df_pass2 = df_pass1.copy()

# Identify meaningful contributors
contributors = df_pass2[df_pass2['games_played'] >= CONTRIBUTOR_THRESHOLD].copy()

# For each draft year, compress the pick numbers and re-rank by performance score
compressed_results = []

for year in contributors['draft_year'].unique():
    year_df = contributors[contributors['draft_year'] == year].copy()
    
    # Re-number actual pick positions 1 through N in draft order
    year_df = year_df.sort_values('pick_number')
    year_df['compressed_pick'] = range(1, len(year_df) + 1)
    
    # Re-rank by performance score 1 through N
    year_df['compressed_hindsight'] = year_df['performance_score'].rank(
        ascending=False, method='min'
    ).astype(int)
    
    # New plus/minus in compressed space
    year_df['compressed_plus_minus'] = year_df['compressed_pick'] - year_df['compressed_hindsight']
    
    compressed_results.append(year_df)

contributors_compressed = pd.concat(compressed_results)

# Update plus/minus for contributors in df_pass2
df_pass2 = df_pass2.merge(
    contributors_compressed[['player_name', 'draft_year', 'compressed_plus_minus']],
    on=['player_name', 'draft_year'],
    how='left'
)

df_pass2.loc[df_pass2['compressed_plus_minus'].notna(), 'plus_minus'] = \
    df_pass2.loc[df_pass2['compressed_plus_minus'].notna(), 'compressed_plus_minus']

df_pass2 = df_pass2.drop(columns=['compressed_plus_minus', 'nhl_player_count'])

# Quick sanity check
print(f"Total players: {len(df_pass2)}")
print(f"Contributors re-ranked: {len(contributors_compressed)}")
print(f"Plus/minus range after Pass 2: {df_pass2['plus_minus'].min()} to {df_pass2['plus_minus'].max()}")

Total players: 2484
Contributors re-ranked: 646
Plus/minus range after Pass 2: -95.0 to 47.0


In [22]:
# Check Hornqvist and a few other known steals
steals_check = df_pass2[df_pass2['player_name'].isin([
    'Patric Hornqvist', 'Anton Stralman', 'Ondrej Palat', 
    'Mark Stone', 'Erik Condra'
])][['player_name', 'draft_year', 'pick_number', 'games_played', 'performance_score', 'plus_minus']]

print(steals_check.to_string())

# Top 15 plus/minus after Pass 2
print("\nTop 15 plus/minus after Pass 2:")
print(df_pass2.nlargest(15, 'plus_minus')[
    ['player_name', 'drafted_by', 'draft_year', 'round', 'games_played', 'performance_score', 'plus_minus']
].to_string())

           player_name  draft_year  pick_number  games_played  performance_score  plus_minus
193     Anton Stralman        2005          216         938.0            1608.97        29.0
206   Patric Hornqvist        2005          230         901.0            2144.47        34.0
391        Erik Condra        2006          211         372.0             598.71         5.0
1122        Mark Stone        2010          178         750.0            2339.26        47.0
1339      Ondrej Palat        2011          208         889.0            2077.51        42.0

Top 15 plus/minus after Pass 2:
              player_name    drafted_by  draft_year  round  games_played  performance_score  plus_minus
1122           Mark Stone        Ottawa        2010      6         750.0            2339.26        47.0
1339         Ondrej Palat     Tampa Bay        2011      7         889.0            2077.51        42.0
1247      Johnny Gaudreau       Calgary        2011      4         763.0            2464.47      

In [23]:
# Before and after team rankings comparison
before = (df.groupby('drafted_by')['plus_minus']
          .sum()
          .reset_index()
          .rename(columns={'plus_minus': 'plus_minus_before'}))
before['rank_before'] = before['plus_minus_before'].rank(ascending=False).astype(int)

after = (df_pass2.groupby('drafted_by')['plus_minus']
         .sum()
         .reset_index()
         .rename(columns={'plus_minus': 'plus_minus_after'}))
after['rank_after'] = after['plus_minus_after'].rank(ascending=False).astype(int)

comparison = before.merge(after, on='drafted_by')
comparison['rank_change'] = comparison['rank_before'] - comparison['rank_after']
comparison['points_change'] = comparison['plus_minus_after'] - comparison['plus_minus_before']

comparison = comparison.sort_values('rank_after')
print(comparison.to_string())

      drafted_by  plus_minus_before  rank_before  plus_minus_after  rank_after  rank_change  points_change
28         Vegas                252           31             -32.0           1           30         -284.0
13   Los Angeles               3940            2            -326.0           2            0        -4266.0
10       Detroit               3536            3            -386.0           3            0        -3922.0
18     Nashville               3079            9            -405.0           4            5        -3484.0
14     Minnesota               2871           11            -410.0           5            6        -3281.0
19    New Jersey               3127            8            -422.0           6            2        -3549.0
23      San Jose               4077            1            -432.0           7           -6        -4509.0
21  Philadelphia               2871           11            -455.0           8            3        -3326.0
20        Ottawa               2775  

In [30]:
team = 'Boston'

team_breakdown = df_pass2[df_pass2['drafted_by'] == team][
    ['player_name', 'draft_year', 'round', 'pick_number', 'performance_score', 'plus_minus']
].sort_values('plus_minus', ascending=False)

print(f"{team} — full draft breakdown sorted by plus/minus:")
print(f"Total plus/minus: {team_breakdown['plus_minus'].sum()}")
print(team_breakdown.to_string())

Boston — full draft breakdown sorted by plus/minus:
Total plus/minus: -717.0
                   player_name  draft_year  round  pick_number  performance_score  plus_minus
270              Brad Marchand        2006      3           71            3519.86        26.0
1743            David Pastrnak        2014      1           25            2902.19        20.0
1821             Danton Heinen        2014      4          116            1159.34        19.0
250                Milan Lucic        2006      2           50            2518.94        15.0
1770               Ryan Donato        2014      2           56            1129.66         8.0
1498              Matt Benning        2012      6          175             698.58         8.0
96            Vladimir Sobotka        2005      4          106             939.59         6.0
1416             Matt Grzelcyk        2012      3           85            1020.23         5.0
2142             Ryan Lindgren        2016      2           49             71

In [29]:
year_contributors_2016 = contributors_compressed[contributors_compressed['draft_year'] == 2016]

print(f"2016 contributors: {len(year_contributors_2016)}")
print(f"Sum of compressed plus/minus: {year_contributors_2016['compressed_plus_minus'].sum()}")
print(f"Positive compressed plus/minus: {year_contributors_2016[year_contributors_2016['compressed_plus_minus'] > 0]['compressed_plus_minus'].sum()}")
print(f"Negative compressed plus/minus: {year_contributors_2016[year_contributors_2016['compressed_plus_minus'] < 0]['compressed_plus_minus'].sum()}")

2016 contributors: 43
Sum of compressed plus/minus: 0
Positive compressed plus/minus: 189
Negative compressed plus/minus: -189


In [31]:
# Get the minimum compressed plus/minus per draft year
# This is the floor of the compressed scale for each year
compressed_floor = contributors_compressed.groupby('draft_year')['compressed_plus_minus'].min().reset_index()
compressed_floor.columns = ['draft_year', 'compressed_floor']

print("Compressed floor per draft year:")
print(compressed_floor.to_string())

Compressed floor per draft year:
    draft_year  compressed_floor
0         2005               -36
1         2006               -23
2         2007               -35
3         2008               -27
4         2009               -43
5         2010               -37
6         2011               -32
7         2012               -47
8         2013               -28
9         2014               -43
10        2015               -41
11        2016               -26
12        2017               -39


In [32]:
# Merge the compressed floor onto df_pass2
df_pass2 = df_pass2.merge(compressed_floor, on='draft_year', how='left')

# Cap non-contributor negatives at the compressed floor for their draft year
mask_non_contributor_negatives = (
    (df_pass2['games_played'] < GP_THRESHOLD) &
    (df_pass2['plus_minus'] < 0)
)

df_pass2.loc[mask_non_contributor_negatives, 'plus_minus'] = df_pass2.loc[
    mask_non_contributor_negatives, ['plus_minus', 'compressed_floor']
].max(axis=1)

df_pass2 = df_pass2.drop(columns=['compressed_floor'])

print(f"Plus/minus range after capping non-contributor negatives:")
print(f"Min: {df_pass2['plus_minus'].min()}")
print(f"Max: {df_pass2['plus_minus'].max()}")

# Show updated team rankings
after_final = (df_pass2.groupby('drafted_by')['plus_minus']
               .sum()
               .reset_index()
               .rename(columns={'plus_minus': 'total_plus_minus'}))
after_final['rank'] = after_final['total_plus_minus'].rank(ascending=False).astype(int)
after_final = after_final.sort_values('rank')

print("\nFinal team rankings:")
print(after_final.to_string())

Plus/minus range after capping non-contributor negatives:
Min: -47.0
Max: 47.0

Final team rankings:
      drafted_by  total_plus_minus  rank
28         Vegas             -32.0     1
13   Los Angeles            -248.0     2
14     Minnesota            -266.0     3
21  Philadelphia            -286.0     4
23      San Jose            -286.0     4
29    Washington            -290.0     6
18     Nashville            -348.0     7
10       Detroit            -361.0     8
17    NY Rangers            -380.0     9
19    New Jersey            -390.0    10
20        Ottawa            -406.0    11
22    Pittsburgh            -420.0    12
5       Carolina            -445.0    13
9         Dallas            -454.0    14
27     Vancouver            -487.0    15
2         Boston            -499.0    16
4        Calgary            -507.0    17
30      Winnipeg            -524.0    18
8       Columbus            -530.0    19
24     St. Louis            -533.0    20
25     Tampa Bay            -537.0    

In [37]:
team = 'Edmonton'

team_breakdown = df_pass2[df_pass2['drafted_by'] == team][
    ['player_name', 'draft_year', 'round', 'pick_number', 'games_played', 'performance_score', 'plus_minus']
].sort_values('plus_minus', ascending=False)

print(f"{team} — full draft breakdown sorted by plus/minus:")
print(f"Total plus/minus: {team_breakdown['plus_minus'].sum()}")
print(team_breakdown.to_string())

Edmonton — full draft breakdown sorted by plus/minus:
Total plus/minus: -684.0
              player_name  draft_year  round  pick_number  games_played  performance_score  plus_minus
2047          John Marino        2015      6          154         429.0             774.79        19.0
605         Jordan Eberle        2008      1           22        1122.0            2889.88        12.0
1423      Erik Gustafsson        2012      4           93         517.0            1066.60        10.0
246            Jeff Petry        2006      2           45        1039.0            1938.97         9.0
22        Andrew Cogliano        2005      1           25        1294.0            2356.56         7.0
1257        Tobias Rieder        2011      4          114         478.0             810.05         7.0
1721       Leon Draisaitl        2014      1            3         853.0            3259.79         2.0
2234       Aapeli Rasanen        2016      6          153           0.0               0.00       

In [45]:
# import pandas as pd

# df = pd.read_csv('../data/outputs/draft_scores_pre_fix.csv')

# # Exclude goalies (already excluded in formula.ipynb so should be fine)
# # For each draft year, calculate percentiles
# results = []

# for year in df['draft_year'].unique():
#     year_df = df[df['draft_year'] == year].copy()
#     total = len(year_df)
    
#     # Pick percentile — where in the draft were they selected
#     year_df['pick_pct'] = year_df['pick_number'] / total
    
#     # Player percentile — where do they rank by performance score
#     year_df['player_pct'] = year_df['performance_score'].rank(ascending=False, method='average') / total
    
# #     # Players with 0 performance score get player percentile of 1.0
# #     year_df.loc[year_df['performance_score'] == 0, 'player_pct'] = 1.0
    
# #     # Optional: players under X games played also get 1.0
# #     year_df.loc[year_df['games_played'] < 50, 'player_pct'] = 1.0
    
#     # Score
#     year_df['draft_score'] = year_df['pick_pct'] - year_df['player_pct']
    
#     results.append(year_df)

# df_pct = pd.concat(results)

# # Team totals
# team_totals = (df_pct.groupby('drafted_by')['draft_score']
#                .sum()
#                .reset_index()
#                .sort_values('draft_score', ascending=False))
# team_totals['rank'] = range(1, len(team_totals) + 1)

# print(team_totals.to_string())

In [46]:
import pandas as pd

df = pd.read_csv('../data/outputs/draft_scores_pre_fix.csv')

results = []

for year in df['draft_year'].unique():
    year_df = df[df['draft_year'] == year].copy()
    total = len(year_df)
    
    # Pick percentile — where in the draft were they selected
    year_df['pick_pct'] = year_df['pick_number'] / total
    
    # Player percentile — where do they rank by performance score
    year_df['player_pct'] = year_df['performance_score'].rank(ascending=False, method='average') / total
    
    # Draft score
    year_df['draft_score'] = year_df['pick_pct'] - year_df['player_pct']
    
    # Zero performance score players can never contribute a positive score
    year_df.loc[year_df['performance_score'] == 0, 'draft_score'] = year_df.loc[
        year_df['performance_score'] == 0, 'draft_score'
    ].clip(upper=0)
    
    results.append(year_df)

df_pct = pd.concat(results)

# Team totals
team_totals = (df_pct.groupby('drafted_by')['draft_score']
               .sum()
               .reset_index()
               .sort_values('draft_score', ascending=False))
team_totals['rank'] = range(1, len(team_totals) + 1)

print(team_totals.to_string())

      drafted_by  draft_score  rank
13   Los Angeles     7.645233     1
23      San Jose     7.040077     2
21  Philadelphia     4.835773     3
26       Toronto     4.159567     4
25     Tampa Bay     3.784165     5
20        Ottawa     3.780192     6
2         Boston     3.023117     7
8       Columbus     2.936417     8
29    Washington     2.779852     9
19    New Jersey     2.403953    10
18     Nashville     1.955617    11
22    Pittsburgh     1.843002    12
14     Minnesota     1.179662    13
10       Detroit     1.067726    14
9         Dallas     0.868876    15
28         Vegas     0.428571    16
16  NY Islanders     0.332993    17
5       Carolina    -0.344383    18
27     Vancouver    -0.936451    19
17    NY Rangers    -1.064940    20
4        Calgary    -1.122532    21
30      Winnipeg    -1.437985    22
0        Anaheim    -1.597455    23
11      Edmonton    -1.781114    24
24     St. Louis    -2.163140    25
7       Colorado    -2.716662    26
12       Florida    -2.86305

In [54]:
team = 'New Jersey'

team_breakdown = df_pct[df_pct['drafted_by'] == team][
    ['player_name', 'draft_year', 'round', 'games_played', 
     'performance_score', 'pick_pct', 'player_pct', 'draft_score']
].sort_values('draft_score', ascending=False)

print(f"{team} — full draft breakdown sorted by draft score:")
print(f"Total draft score: {round(team_breakdown['draft_score'].sum(), 3)}")
print(team_breakdown.to_string())

New Jersey — full draft breakdown sorted by draft score:
Total draft score: 2.404
                player_name  draft_year  round  games_played  performance_score  pick_pct  player_pct  draft_score
2241           Jesper Bratt        2016      6         617.0            1755.13  0.839378    0.025907     0.813472
1476           Alex Kerfoot        2012      5         623.0            1298.55  0.802139    0.128342     0.673797
138              Mark Fayne        2005      5         389.0             537.85  0.748792    0.178744     0.570048
2268          Jeremy Davies        2016      7          23.0              29.87  0.994819    0.445596     0.549223
333          Olivier Magnan        2006      5          18.0              18.00  0.791444    0.326203     0.465241
2050            Brett Seney        2015      6          66.0              98.06  0.839572    0.422460     0.417112
501          Matt Halischuk        2007      4         280.0             451.75  0.612565    0.198953     0.41361

In [55]:
import pandas as pd

df = pd.read_csv('../data/outputs/draft_scores_pre_fix.csv')

GP_THRESHOLD = 200
results = []

for year in df['draft_year'].unique():
    year_df = df[df['draft_year'] == year].copy()
    total = len(year_df)
    
    year_df['pick_pct'] = year_df['pick_number'] / total
    year_df['player_pct'] = year_df['performance_score'].rank(ascending=False, method='average') / total
    year_df['draft_score'] = year_df['pick_pct'] - year_df['player_pct']
    
    # Non-contributors can never give their team a positive score
    year_df.loc[year_df['games_played'] < GP_THRESHOLD, 'draft_score'] = year_df.loc[
        year_df['games_played'] < GP_THRESHOLD, 'draft_score'
    ].clip(upper=0)
    
    results.append(year_df)

df_pct = pd.concat(results)

# Team totals
team_totals = (df_pct.groupby('drafted_by')['draft_score']
               .sum()
               .reset_index()
               .sort_values('draft_score', ascending=False))
team_totals['rank'] = range(1, len(team_totals) + 1)

# Normalize so worst team = 0
team_totals['draft_score_normalized'] = team_totals['draft_score'] - team_totals['draft_score'].min()

print(team_totals.to_string())

      drafted_by  draft_score  rank  draft_score_normalized
13   Los Angeles     3.433527     1               10.940983
20        Ottawa     1.185870     2                8.693325
23      San Jose     0.417028     3                7.924484
28         Vegas    -0.209184     4                7.298271
26       Toronto    -0.240909     5                7.266546
8       Columbus    -0.482269     6                7.025186
22    Pittsburgh    -1.057827     7                6.449628
19    New Jersey    -1.172483     8                6.334972
18     Nashville    -1.337669     9                6.169786
14     Minnesota    -1.635459    10                5.871996
21  Philadelphia    -1.771019    11                5.736436
2         Boston    -2.272778    12                5.234677
10       Detroit    -2.398611    13                5.108844
29    Washington    -2.560366    14                4.947090
25     Tampa Bay    -2.626730    15                4.880725
17    NY Rangers    -2.673652    16     

In [58]:
team = 'NY Rangers'

team_breakdown = df_pct[df_pct['drafted_by'] == team][
    ['player_name', 'draft_year', 'round', 'games_played', 
     'performance_score', 'pick_pct', 'player_pct', 'draft_score']
].sort_values('draft_score', ascending=False)

print(f"{team} — full draft breakdown sorted by draft score:")
print(f"Total draft score: {round(team_breakdown['draft_score'].sum(), 3)}")
print(team_breakdown.to_string())

NY Rangers — full draft breakdown sorted by draft score:
Total draft score: -2.674
                 player_name  draft_year  round  games_played  performance_score  pick_pct  player_pct  draft_score
546             Carl Hagelin        2007      6         713.0            1390.84  0.879581    0.120419     0.759162
2443           Morgan Barron        2017      6         311.0             494.20  0.887755    0.163265     0.724490
1105             Jesper Fast        2010      6         703.0            1270.92  0.830688    0.132275     0.698413
684               Dale Weise        2008      4         513.0             799.25  0.590426    0.170213     0.420213
1628             Ryan Graves        2013      4         452.0             724.51  0.578947    0.200000     0.378947
97                 Tom Pyatt        2005      4         445.0             676.29  0.516908    0.159420     0.357488
1597        Pavel Buchnevich        2013      3         658.0            1798.42  0.394737    0.047368   

In [57]:
team = 'Ottawa'
team_breakdown = df_pct[df_pct['drafted_by'] == team][
    ['player_name', 'draft_year', 'round', 'pick_number', 'games_played', 'performance_score', 'draft_score']
].sort_values('draft_score', ascending=False)

print(f"Total: {round(team_breakdown['draft_score'].sum(), 3)}")
print(team_breakdown.to_string())

Total: 1.186
              player_name  draft_year  round  pick_number  games_played  performance_score  draft_score
391           Erik Condra        2006      7          211         372.0             598.71     0.946524
1122           Mark Stone        2010      6          178         750.0            2339.26     0.899471
1335         Ryan Dzingel        2011      7          204         404.0             834.52     0.869110
181        Colin Greening        2005      7          204         286.0             519.58     0.801932
891          Mike Hoffman        2009      5          130         745.0            1860.23     0.592593
2395      Drake Batherson        2017      4          121         452.0            1253.50     0.571429
709       Mark Borowiecki        2008      5          139         458.0             586.24     0.547872
691           Derek Grant        2008      4          119         427.0             729.28     0.452128
1239  Jean-Gabriel Pageau        2011      4       

## Weighted inverse test

In [61]:
import pandas as pd

df = pd.read_csv('../data/outputs/draft_scores_pre_fix.csv')

GP_THRESHOLD = 200
WEIGHT = 0.3
results = []

for year in df['draft_year'].unique():
    year_df = df[df['draft_year'] == year].copy()
    total = len(year_df)
    
    year_df['pick_pct'] = year_df['pick_number'] / total
    year_df['player_pct'] = year_df['performance_score'].rank(ascending=False, method='average') / total
    
    # Lightly weighted by inverse pick percentile
    year_df['draft_score'] = (year_df['pick_pct'] - year_df['player_pct']) * (1 - (year_df['pick_pct'] * WEIGHT))
    
    # Non-contributors cannot give positive scores
    year_df.loc[year_df['games_played'] < GP_THRESHOLD, 'draft_score'] = year_df.loc[
        year_df['games_played'] < GP_THRESHOLD, 'draft_score'
    ].clip(upper=0)
    
    results.append(year_df)

df_pct = pd.concat(results)

team_totals = (df_pct.groupby('drafted_by')['draft_score']
               .sum()
               .reset_index()
               .sort_values('draft_score', ascending=False))
team_totals['rank'] = range(1, len(team_totals) + 1)
team_totals['draft_score_normalized'] = team_totals['draft_score'] - team_totals['draft_score'].min()

print(team_totals.to_string())

      drafted_by  draft_score  rank  draft_score_normalized
13   Los Angeles     1.910814     1                8.790501
20        Ottawa     0.037176     2                6.916863
28         Vegas    -0.189704     3                6.689984
23      San Jose    -0.785043     4                6.094645
8       Columbus    -1.307094     5                5.572594
19    New Jersey    -1.410292     6                5.469395
22    Pittsburgh    -1.417427     7                5.462261
26       Toronto    -1.529004     8                5.350683
18     Nashville    -1.565753     9                5.313934
14     Minnesota    -1.880179    10                4.999508
21  Philadelphia    -2.010829    11                4.868858
10       Detroit    -2.289204    12                4.590483
2         Boston    -2.456655    13                4.423032
17    NY Rangers    -2.797974    14                4.081713
29    Washington    -2.812932    15                4.066755
25     Tampa Bay    -2.959204    16     

In [62]:
team = 'Ottawa'
team_breakdown = df_pct[df_pct['drafted_by'] == team][
    ['player_name', 'draft_year', 'round', 'pick_number', 'games_played', 'performance_score', 'draft_score']
].sort_values('draft_score', ascending=False)

print(f"Total: {round(team_breakdown['draft_score'].sum(), 3)}")
print(team_breakdown.to_string())

Total: 0.037
              player_name  draft_year  round  pick_number  games_played  performance_score  draft_score
1122           Mark Stone        2010      6          178         750.0            2339.26     0.645335
391           Erik Condra        2006      7          211         372.0             598.71     0.626123
1335         Ryan Dzingel        2011      7          204         404.0             834.52     0.590631
181        Colin Greening        2005      7          204         286.0             519.58     0.564839
891          Mike Hoffman        2009      5          130         745.0            1860.23     0.470312
2395      Drake Batherson        2017      4          121         452.0            1253.50     0.465598
709       Mark Borowiecki        2008      5          139         458.0             586.24     0.426350
691           Derek Grant        2008      4          119         427.0             729.28     0.366272
1239  Jean-Gabriel Pageau        2011      4       

In [71]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/outputs/draft_scores_pre_fix.csv')

# WEIGHT = 0.3
results = []

for year in sorted(df['draft_year'].unique()):
    year_df = df[df['draft_year'] == year].copy()
    total = len(year_df)
    
    # Pick percentile
    year_df['pick_pct'] = (year_df['pick_number'] - 1) / (total - 1)
    
    # Log transform performance score then rank
    # log1p handles zeros gracefully (log(0+1) = 0)
    # method='max' means all tied zeros get player_pct of 1.0 naturally
    year_df['log_score'] = np.log1p(year_df['performance_score'])
    year_df['player_pct'] = year_df['log_score'].rank(ascending=False, method='max') / total
    
#     # Draft score with light weighting
#     year_df['draft_score'] = (year_df['pick_pct'] - year_df['player_pct']) * (1 - (year_df['pick_pct'] * WEIGHT))
    year_df['draft_score'] = year_df['pick_pct'] - year_df['player_pct']
    
    results.append(year_df)

df_pct = pd.concat(results)

# Team totals
team_totals = (df_pct.groupby('drafted_by')['draft_score']
               .sum()
               .reset_index()
               .sort_values('draft_score', ascending=False))
team_totals['rank'] = range(1, len(team_totals) + 1)
team_totals['draft_score_normalized'] = team_totals['draft_score'] - team_totals['draft_score'].min()

print(team_totals.to_string())

      drafted_by  draft_score  rank  draft_score_normalized
13   Los Angeles     1.972367     1               15.249333
21  Philadelphia     0.761329     2               14.038295
23      San Jose     0.270105     3               13.547070
28         Vegas    -0.320879     4               12.956086
2         Boston    -0.352610     5               12.924355
20        Ottawa    -1.621980     6               11.654985
25     Tampa Bay    -2.183591     7               11.093375
19    New Jersey    -2.340661     8               10.936304
8       Columbus    -3.494822     9                9.782143
22    Pittsburgh    -3.597290    10                9.679676
26       Toronto    -3.714477    11                9.562489
29    Washington    -3.894688    12                9.382277
18     Nashville    -4.039638    13                9.237328
14     Minnesota    -4.396122    14                8.880843
10       Detroit    -5.272739    15                8.004227
9         Dallas    -6.087249    16     

In [72]:
# Team breakdown
team = 'New Jersey'
team_breakdown = df_pct[df_pct['drafted_by'] == team][
    ['player_name', 'draft_year', 'round', 'games_played', 
     'performance_score', 'pick_pct', 'player_pct', 'draft_score']
].sort_values('draft_score', ascending=False)

print(f"\n{team} — full draft breakdown sorted by draft score:")
print(f"Total draft score: {round(team_breakdown['draft_score'].sum(), 3)}")
print(team_breakdown.to_string())


New Jersey — full draft breakdown sorted by draft score:
Total draft score: -2.341
                player_name  draft_year  round  games_played  performance_score  pick_pct  player_pct  draft_score
2241           Jesper Bratt        2016      6         617.0            1755.13  0.838542    0.025907     0.812635
1476           Alex Kerfoot        2012      5         623.0            1298.55  0.801075    0.128342     0.672733
138              Mark Fayne        2005      5         389.0             537.85  0.747573    0.178744     0.568829
2268          Jeremy Davies        2016      7          23.0              29.87  0.994792    0.445596     0.549196
333          Olivier Magnan        2006      5          18.0              18.00  0.790323    0.326203     0.464119
2050            Brett Seney        2015      6          66.0              98.06  0.838710    0.422460     0.416250
501          Matt Halischuk        2007      4         280.0             451.75  0.610526    0.198953     0.411

In [79]:
## GPTs
import pandas as pd
import numpy as np

df = pd.read_csv('../data/outputs/draft_scores_pre_fix.csv')

results = []

for year in sorted(df['draft_year'].unique()):
    year_df = df[df['draft_year'] == year].copy()
    total = len(year_df)
    
    # Pick percentile (scaled 0 to 1)
    year_df['pick_pct'] = (year_df['pick_number'] - 1) / (total - 1)
    
    # Log transform performance score
    year_df['log_score'] = np.log1p(year_df['performance_score'])
    
    # 🔥 NEW: Smooth games played weighting (Option 2)
#     year_df['gp_weight'] = (
#         year_df['games_played'] / (year_df['games_played'] + 50)
#     )

    year_df['gp_weight'] = 1 - np.exp(-year_df['games_played'] / 40)
    
    # Adjusted score (reduces impact of very low GP players)
    year_df['adj_score'] = year_df['log_score'] * year_df['gp_weight']
    
    # Rank based on adjusted score
    year_df['player_pct'] = (
        year_df['adj_score']
        .rank(ascending=False, method='max') / total
    )
    
    # Draft score
    year_df['draft_score'] = year_df['pick_pct'] - year_df['player_pct']
    
    results.append(year_df)

# Combine all drafts
df_pct = pd.concat(results)

# Team performance (average = efficiency per pick)
team_totals = (
    df_pct.groupby('drafted_by')['draft_score']
    .mean()
    .reset_index()
    .sort_values('draft_score', ascending=False)
)

# Rank teams
team_totals['rank'] = range(1, len(team_totals) + 1)

# Optional: shifted version for visualization
team_totals['draft_score_shifted'] = (
    team_totals['draft_score'] - team_totals['draft_score'].min()
)

print(team_totals.to_string(index=False))

  drafted_by  draft_score  rank  draft_score_shifted
 Los Angeles     0.022290     1             0.184523
Philadelphia     0.009840     2             0.172073
    San Jose     0.003670     3             0.165903
      Boston    -0.004782     4             0.157451
      Ottawa    -0.020261     5             0.141972
   Tampa Bay    -0.025390     6             0.136843
  New Jersey    -0.028038     7             0.134195
       Vegas    -0.030557     8             0.131676
    Columbus    -0.041182     9             0.121051
     Toronto    -0.042985    10             0.119248
   Nashville    -0.047355    11             0.114878
  Washington    -0.050610    12             0.111623
  Pittsburgh    -0.051188    13             0.111045
   Minnesota    -0.060076    14             0.102158
     Detroit    -0.060850    15             0.101383
NY Islanders    -0.074728    16             0.087505
      Dallas    -0.081028    17             0.081205
   Vancouver    -0.084209    18             0.

In [80]:
# Team breakdown
team = 'New Jersey'
team_breakdown = df_pct[df_pct['drafted_by'] == team][
    ['player_name', 'draft_year', 'round', 'games_played', 
     'performance_score', 'pick_pct', 'player_pct', 'draft_score']
].sort_values('draft_score', ascending=False)

print(f"\n{team} — full draft breakdown sorted by draft score:")
print(f"Total draft score: {round(team_breakdown['draft_score'].sum(), 3)}")
print(team_breakdown.to_string())


New Jersey — full draft breakdown sorted by draft score:
Total draft score: -2.355
                player_name  draft_year  round  games_played  performance_score  pick_pct  player_pct  draft_score
2241           Jesper Bratt        2016      6         617.0            1755.13  0.838542    0.025907     0.812635
1476           Alex Kerfoot        2012      5         623.0            1298.55  0.801075    0.128342     0.672733
138              Mark Fayne        2005      5         389.0             537.85  0.747573    0.178744     0.568829
2268          Jeremy Davies        2016      7          23.0              29.87  0.994792    0.445596     0.549196
333          Olivier Magnan        2006      5          18.0              18.00  0.790323    0.315508     0.474815
2050            Brett Seney        2015      6          66.0              98.06  0.838710    0.422460     0.416250
501          Matt Halischuk        2007      4         280.0             451.75  0.610526    0.198953     0.411

In [83]:
## GPTs
import pandas as pd
import numpy as np

df = pd.read_csv('../data/outputs/draft_scores_pre_fix.csv')

results = []

for year in sorted(df['draft_year'].unique()):
    year_df = df[df['draft_year'] == year].copy()
    
    # Use actual max pick number to avoid pick_pct > 1
    max_pick = year_df['pick_number'].max()
    year_df['pick_pct'] = (year_df['pick_number'] - 1) / (max_pick - 1)
    
    # Log transform performance score
    year_df['log_score'] = np.log1p(year_df['performance_score'])
    
    # Smooth games played weighting (Option 2)
    year_df['gp_weight'] = 1 - np.exp(-year_df['games_played'] / 40)
    
    # Adjusted score (reduces impact of very low GP players)
    year_df['adj_score'] = year_df['log_score'] * year_df['gp_weight']
    
    # Rank based on adjusted score
    total = len(year_df)
    year_df['player_pct'] = (
        year_df['adj_score']
        .rank(ascending=False, method='max') / total
    )
    
    # Draft score
    year_df['draft_score'] = year_df['pick_pct'] - year_df['player_pct']
    
    results.append(year_df)

# Combine all drafts
df_pct = pd.concat(results)

# Team performance (average = efficiency per pick)
team_totals = (
    df_pct.groupby('drafted_by')['draft_score']
    .mean()
    .reset_index()
    .sort_values('draft_score', ascending=False)
)

# Rank teams
team_totals['rank'] = range(1, len(team_totals) + 1)

# Optional: shifted version for visualization
team_totals['draft_score_shifted'] = (
    team_totals['draft_score'] - team_totals['draft_score'].min()
)

print(team_totals.to_string(index=False))

  drafted_by  draft_score  rank  draft_score_shifted
 Los Angeles    -0.038008     1             0.173708
Philadelphia    -0.047435     2             0.164280
      Boston    -0.060825     3             0.150891
    San Jose    -0.061930     4             0.149785
       Vegas    -0.070493     5             0.141222
      Ottawa    -0.075970     6             0.135745
   Tampa Bay    -0.083510     7             0.128205
  New Jersey    -0.086473     8             0.125242
    Columbus    -0.097206     9             0.114509
     Toronto    -0.101776    10             0.109939
   Nashville    -0.105123    11             0.106592
  Pittsburgh    -0.111221    12             0.100494
  Washington    -0.112591    13             0.099124
   Minnesota    -0.120385    14             0.091330
     Detroit    -0.122299    15             0.089417
NY Islanders    -0.130720    16             0.080995
      Dallas    -0.133585    17             0.078130
    Carolina    -0.139753    18             0.

In [84]:
# Team breakdown
team = 'New Jersey'
team_breakdown = df_pct[df_pct['drafted_by'] == team][
    ['player_name', 'draft_year', 'round', 'games_played', 
     'performance_score', 'pick_pct', 'player_pct', 'draft_score']
].sort_values('draft_score', ascending=False)

print(f"\n{team} — full draft breakdown sorted by draft score:")
print(f"Total draft score: {round(team_breakdown['draft_score'].sum(), 3)}")
print(team_breakdown.to_string())


New Jersey — full draft breakdown sorted by draft score:
Total draft score: -7.264
                player_name  draft_year  round  games_played  performance_score  pick_pct  player_pct  draft_score
2241           Jesper Bratt        2016      6         617.0            1755.13  0.766667    0.025907     0.740760
1476           Alex Kerfoot        2012      5         623.0            1298.55  0.709524    0.128342     0.581182
138              Mark Fayne        2005      5         389.0             537.85  0.672489    0.178744     0.493745
2268          Jeremy Davies        2016      7          23.0              29.87  0.909524    0.445596     0.463928
333          Olivier Magnan        2006      5          18.0              18.00  0.693396    0.315508     0.377888
501          Matt Halischuk        2007      4         280.0             451.75  0.552381    0.198953     0.353428
659           Adam Henrique        2008      3        1042.0            2340.43  0.385714    0.047872     0.337